# 电池状态与寿命预测：逐步实验

先按README安装依赖，再用Jupyter打开。所有数据均为程序构造；本笔记不训练长时域RUL模型。

In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
root = Path.cwd()
if root.name == "notebooks": root = root.parent
assert (root / "chapters/ch02/run.py").exists(), "请从仓库根目录或notebooks目录启动"
sys.path.insert(0, str(root / "chapters/ch02"))
import run


## 1. 生成记录并检查窗口

修改未来目标前先验证输入范围。SOH为容量与2 A·h参考容量之比。

In [ ]:
rows = run.generate()
x, y, meta = run.samples(rows)
print("输入形状:", x.shape, "标签形状:", y.shape)
print("第一个窗口:", x[0])
print("首个目标:", meta[0], y[0])
assert meta[0]["last_input_cycle"] < meta[0]["target_cycle"]


## 2. 对照SOC误差

先手算电流偏高0.02 A时，60分钟后累计误差。

In [ ]:
soc_rows, soc_metrics = run.soc()
print(json.dumps(soc_metrics, ensure_ascii=False, indent=2))
assert abs(soc_metrics["current_bias"]["final_error_pp"] + .02/1.8*100) < 1e-10


## 3. 从头训练并冻结评测

自动选一个全新输出目录；训练、选择、校准和测试依次执行。候选参数位于config.json。

In [ ]:
import tempfile
output = Path(tempfile.mkdtemp(prefix="battery_lesson_")) / "run"
metrics = run.run(output)
print("输出目录:", output)
print("验证集选中:", metrics["selected_family"])


## 4. 从本次结果重新绘图

先安装requirements-viz.txt与中文字体。图像文件直接来自上一步结果。

In [ ]:
import plot_results
from IPython.display import display, Image
figure_dir = output / "figures"
plot_results.build_all(output, figure_dir)
display(Image(filename=str(figure_dir / "fig02_08_interval.png"), width=900))


## 5. 解释边界

比较岭回归与网络的误差差异、四个电芯覆盖率、内阻偏移压力测试。不要把压力测试称为重新训练后的消融。

In [ ]:
print(json.dumps({"test": {k:v["test"] for k,v in metrics["scores"].items()}, "interval": metrics["empirical_interval"], "sensor_shift": metrics["sensor_offset_test"]}, ensure_ascii=False, indent=2))
